# NB6 — PCBS Bridge Scoring

**Purpose:** Integrate all five evidence components — P(g) PMOS, T(g) Transition,
C(g) Cancer, D(g) Directional consistency, K(g) Independent knowledge — into the
final **PCOS–Cancer Bridge Score (PCBS)** for every gene in the analysis universe.

| Component | Source Notebook | File |
|-----------|---------------|------|
| P(g) PMOS evidence | NB1 (GSE48301) | `results/all_genes_limma_gse48301.csv` |
| T(g) Transition evidence | NB3 (GSE106191) | `results/all_genes_limma_gse106191.csv` |
| C(g) Cancer evidence | NB2 (TCGA-UCEC) | `results/all_genes_limma_tcga_ucec.csv` |
| D(g) Directional consistency | NB4 (GSE39099) | `results/directional_support_gse39099.csv` |
| K(g) Independent knowledge | NB5 (MR/GWAS) | `results/kg_scores_mr.csv` |

**PCBS Formula:**

$$PCBS(g) = w_P \cdot P(g) + w_T \cdot T(g) + w_C \cdot C(g) + w_D \cdot |D(g)| + w_K \cdot K(g)$$

All components normalised to [0, 1]; default weights $w_i = 0.20$ (equal contribution).

**Output:** `results/pcbs_scores.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os, warnings
warnings.filterwarnings('ignore')

os.makedirs('results', exist_ok=True)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 100,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

print('NB6 \u2014 PCBS Bridge Scoring')
print('=' * 40)

## 1 \u00b7 Load NB1\u2013NB5 Results

In [ ]:
from google.colab import files
import io

# ---- Upload all five result CSVs from your local machine ----
file_map = {
    'all_genes_limma_gse48301.csv':      'NB1 (PMOS - GSE48301)',
    'all_genes_limma_tcga_ucec.csv':     'NB2 (Cancer - TCGA-UCEC)',
    'all_genes_limma_gse106191.csv':     'NB3 (Transition - GSE106191)',
    'directional_support_gse39099.csv':  'NB4 (Directional - GSE39099)',
    'kg_scores_mr.csv':                  'NB5 (MR/GWAS - K(g))',
}

print("Upload these 5 CSV files when prompted:")
for fname, label in file_map.items():
    print(f"  • {fname}  ←  {label}")
print()

uploaded = files.upload()

# Parse each uploaded file
nb1 = pd.read_csv(io.BytesIO(uploaded['all_genes_limma_gse48301.csv']))
nb2 = pd.read_csv(io.BytesIO(uploaded['all_genes_limma_tcga_ucec.csv']))
nb3 = pd.read_csv(io.BytesIO(uploaded['all_genes_limma_gse106191.csv']))
nb4 = pd.read_csv(io.BytesIO(uploaded['directional_support_gse39099.csv']))
nb5 = pd.read_csv(io.BytesIO(uploaded['kg_scores_mr.csv']))

print('\nData loaded:')
print(f'  NB1 (PMOS - GSE48301):        {len(nb1):,} genes')
print(f'  NB2 (Cancer - TCGA-UCEC):      {len(nb2):,} genes')
print(f'  NB3 (Transition - GSE106191):  {len(nb3):,} genes')
print(f'  NB4 (Directional - GSE39099):  {len(nb4):,} genes')
print(f'  NB5 (MR/GWAS - K(g)):          {len(nb5):,} genes')

# Significant genes per dataset
for name, df in [('NB1 P(g)', nb1), ('NB2 C(g)', nb2), ('NB3 T(g)', nb3)]:
    n_sig = (df['adj_pvalue'] < 0.05).sum()
    print(f'  {name}: {n_sig:,} significant (adj.p < 0.05)')

n_dg = (nb4['D_g'].abs() > 0).sum()
n_kg = (nb5['K_g'] > 0).sum()
print(f'  NB4 D(g): {n_dg:,} genes with |D(g)| > 0')
print(f'  NB5 K(g): {n_kg:,} genes with K(g) > 0')

## 2 \u00b7 Compute Component Scores

**Within-component scoring** for P(g), T(g), C(g):
- Only genes with adj. p-value < 0.05 receive non-zero scores (teacher-approved threshold)
- Score combines effect size |logFC| and statistical significance \u2212log\u2081\u2080(adj.p)
- Formula: `raw = |logFC| \u00d7 min(\u2212log\u2081\u2080(adj.p) / 10, 1.0)`, then min-max normalised to [0, 1]
- No hard logFC cutoff: small-effect genes score low naturally through the continuous formula

**D(g)** and **K(g)** are pre-computed in NB4 and NB5 respectively.

In [ ]:
def compute_component_score(df, logfc_col='logFC', pval_col='adj_pvalue',
                            gene_col='Gene', alpha=0.05, sig_cap=10.0):
    """
    Compute evidence score from limma results.

    Score = |logFC| * min(-log10(adj_pvalue) / sig_cap, 1.0)
    Normalised to [0, 1].

    Parameters
    ----------
    df       : DataFrame with limma output
    alpha    : significance threshold (genes above this get 0)
    sig_cap  : -log10(pval) value that maps to weight 1.0

    Returns
    -------
    DataFrame with Gene and score columns
    """
    result = df[[gene_col]].copy()
    sig_mask = df[pval_col] < alpha

    raw = pd.Series(0.0, index=df.index)
    if sig_mask.any():
        effect = df.loc[sig_mask, logfc_col].abs()
        pvals_clipped = df.loc[sig_mask, pval_col].clip(lower=1e-300)
        sig_weight = np.clip(-np.log10(pvals_clipped) / sig_cap, 0, 1.0)
        raw[sig_mask] = effect * sig_weight

    # Min-max normalise to [0, 1]
    if raw.max() > 0:
        result['score'] = raw / raw.max()
    else:
        result['score'] = 0.0
    return result


# ---- P(g): PMOS evidence from GSE48301 ----
pg = compute_component_score(nb1).rename(columns={'score': 'P_g'})
print(f'P(g) \u2014 PMOS evidence (GSE48301)')
print(f'  Non-zero: {(pg["P_g"] > 0).sum():,} genes')
print(f'  Mean (non-zero): {pg.loc[pg["P_g"]>0, "P_g"].mean():.4f}\n')

# ---- T(g): Transition evidence from GSE106191 ----
tg = compute_component_score(nb3).rename(columns={'score': 'T_g'})
print(f'T(g) \u2014 Transition evidence (GSE106191)')
print(f'  Non-zero: {(tg["T_g"] > 0).sum():,} genes')
print(f'  Mean (non-zero): {tg.loc[tg["T_g"]>0, "T_g"].mean():.4f}\n')

# ---- C(g): Cancer evidence from TCGA-UCEC ----
cg = compute_component_score(nb2).rename(columns={'score': 'C_g'})
print(f'C(g) \u2014 Cancer evidence (TCGA-UCEC)')
print(f'  Non-zero: {(cg["C_g"] > 0).sum():,} genes')
print(f'  Mean (non-zero): {cg.loc[cg["C_g"]>0, "C_g"].mean():.4f}\n')

# ---- D(g): Directional consistency (pre-computed, normalise |D(g)| to [0,1]) ----
dg = nb4[['Gene', 'D_g']].copy()
dg['D_g_raw'] = dg['D_g']  # keep signed version
dg['D_g'] = dg['D_g'].abs()
if dg['D_g'].max() > 0:
    dg['D_g'] = dg['D_g'] / dg['D_g'].max()
print(f'D(g) \u2014 Directional consistency (GSE39099)')
print(f'  Non-zero: {(dg["D_g"] > 0).sum():,} genes')
print(f'  Mean (non-zero): {dg.loc[dg["D_g"]>0, "D_g"].mean():.4f}\n')

# ---- K(g): Independent knowledge (pre-computed, already [0,1]) ----
kg = nb5[['Gene', 'K_g']].copy()
print(f'K(g) \u2014 Independent knowledge (MR/GWAS)')
print(f'  Non-zero: {(kg["K_g"] > 0).sum():,} genes')
print(f'  Mean (non-zero): {kg.loc[kg["K_g"]>0, "K_g"].mean():.4f}')

## 3 \u00b7 Merge All Components

Build a unified gene table by outer-joining all five components on gene symbol.  
Genes absent from a dataset receive 0 for that component (no evidence \u2260 counter-evidence).

In [ ]:
# Start with P(g), then merge T, C, D, K
pcbs_df = pg[['Gene', 'P_g']].copy()

for comp_df, comp_col in [
    (tg[['Gene', 'T_g']], 'T_g'),
    (cg[['Gene', 'C_g']], 'C_g'),
    (dg[['Gene', 'D_g']], 'D_g'),
    (kg[['Gene', 'K_g']], 'K_g'),
]:
    pcbs_df = pcbs_df.merge(comp_df, on='Gene', how='outer')

# Fill NaN with 0 (no evidence from that source)
for col in ['P_g', 'T_g', 'C_g', 'D_g', 'K_g']:
    pcbs_df[col] = pcbs_df[col].fillna(0.0)

pcbs_df = pcbs_df.dropna(subset=['Gene']).drop_duplicates(subset='Gene', keep='first')
pcbs_df = pcbs_df.reset_index(drop=True)

# Count evidence depth
pcbs_df['n_components'] = (pcbs_df[['P_g', 'T_g', 'C_g', 'D_g', 'K_g']] > 0).sum(axis=1)

print(f'Unified gene table: {len(pcbs_df):,} unique genes\n')
print('Genes with evidence per component:')
for col, label in [('P_g','P(g) PMOS'), ('T_g','T(g) Transition'),
                    ('C_g','C(g) Cancer'), ('D_g','D(g) Directional'),
                    ('K_g','K(g) Independent')]:
    n = (pcbs_df[col] > 0).sum()
    print(f'  {label}: {n:,} genes ({100*n/len(pcbs_df):.1f}%)')

print(f'\nEvidence depth:')
for n in range(6):
    count = (pcbs_df['n_components'] == n).sum()
    if count > 0:
        print(f'  {n} components: {count:,} genes')

## 4 \u00b7 Compute PCBS

$$PCBS(g) = 0.20 \cdot P(g) + 0.20 \cdot T(g) + 0.20 \cdot C(g) + 0.20 \cdot D(g) + 0.20 \cdot K(g)$$

Equal weights give each evidence source the same influence. A gene must score highly
across **multiple** sources to rank as a strong bridge candidate.

In [ ]:
# ---- Weights ----
weights = {
    'P_g': 0.20,   # PMOS evidence
    'T_g': 0.20,   # Transition evidence
    'C_g': 0.20,   # Cancer evidence
    'D_g': 0.20,   # Directional consistency
    'K_g': 0.20,   # Independent knowledge
}

# ---- Compute PCBS ----
pcbs_df['PCBS'] = sum(pcbs_df[col] * w for col, w in weights.items())

print('PCBS Score Summary')
print('=' * 45)
print(f'Total genes scored:    {len(pcbs_df):,}')
print(f'Genes with PCBS > 0:  {(pcbs_df["PCBS"] > 0).sum():,}')
print(f'\nDistribution:')
print(pcbs_df['PCBS'].describe().to_string())

for pct in [90, 95, 99]:
    val = pcbs_df['PCBS'].quantile(pct / 100)
    n = (pcbs_df['PCBS'] >= val).sum()
    print(f'  Top {100-pct}% (PCBS >= {val:.4f}): {n:,} genes')

print(f'\nTop 10 Bridge Genes:')
top10 = pcbs_df.nlargest(10, 'PCBS')[['Gene','P_g','T_g','C_g','D_g','K_g','PCBS','n_components']]
print(top10.to_string(index=False, float_format='%.4f'))

## 5 \u00b7 Score Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

components = [
    ('P_g', 'P(g) PMOS',        '#2196F3'),
    ('T_g', 'T(g) Transition',  '#4CAF50'),
    ('C_g', 'C(g) Cancer',      '#F44336'),
    ('D_g', 'D(g) Directional', '#FF9800'),
    ('K_g', 'K(g) Independent', '#9C27B0'),
]

for ax, (col, label, color) in zip(axes.flat[:5], components):
    nonzero = pcbs_df.loc[pcbs_df[col] > 0, col]
    if len(nonzero) > 0:
        ax.hist(nonzero, bins=30, color=color, alpha=0.7, edgecolor='white')
    ax.set_title(f'{label}  (n={len(nonzero):,})')
    ax.set_xlabel('Score')
    ax.set_ylabel('Gene count')

# PCBS
ax = axes[1, 2]
nz = pcbs_df.loc[pcbs_df['PCBS'] > 0, 'PCBS']
ax.hist(nz, bins=50, color='#263238', alpha=0.8, edgecolor='white')
ax.axvline(pcbs_df['PCBS'].quantile(0.95), color='red', ls='--', label='95th pctl')
ax.set_title(f'PCBS  (n={len(nz):,})')
ax.set_xlabel('PCBS Score')
ax.set_ylabel('Gene count')
ax.legend()

plt.suptitle('Component and PCBS Score Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/pcbs_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/pcbs_distributions.png')

## 6 \u00b7 Top Bridge Genes

In [ ]:
top20 = pcbs_df.nlargest(20, 'PCBS').reset_index(drop=True)

print('Top 20 PCBS Bridge Genes')
print('=' * 90)
print(top20[['Gene','P_g','T_g','C_g','D_g','K_g','PCBS','n_components']]
      .to_string(index=False, float_format='%.4f'))

# ---- Horizontal bar chart ----
fig, ax = plt.subplots(figsize=(12, 8))
cmap_dict = {'5':'#D32F2F','4':'#F57C00','3':'#FBC02D','2':'#4CAF50','1':'#2196F3'}
bar_colors = [cmap_dict.get(str(int(n)), '#9E9E9E') for n in top20['n_components']]

y_pos = range(len(top20)-1, -1, -1)
ax.barh(y_pos, top20['PCBS'], color=bar_colors, edgecolor='white')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(top20['Gene'])
ax.set_xlabel('PCBS Score')
ax.set_title('Top 20 Bridge Genes by PCBS\n(colour = number of evidence components)',
             fontweight='bold')

from matplotlib.patches import Patch
handles = [Patch(facecolor=cmap_dict[str(i)], label=f'{i} components')
           for i in [5,4,3,2,1] if str(i) in cmap_dict]
ax.legend(handles=handles, loc='lower right')

plt.tight_layout()
plt.savefig('results/pcbs_top20_genes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/pcbs_top20_genes.png')

## 7 \u00b7 Component Correlation Analysis

Checking whether evidence sources are independent (low correlation) or redundant
(high correlation). Independent sources give more robust bridge rankings.

In [ ]:
active = pcbs_df[pcbs_df['PCBS'] > 0].copy()
comp_cols = ['P_g', 'T_g', 'C_g', 'D_g', 'K_g']
corr_matrix = active[comp_cols].corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            vmin=-0.5, vmax=0.5, ax=axes[0],
            xticklabels=['P(g)','T(g)','C(g)','D(g)','K(g)'],
            yticklabels=['P(g)','T(g)','C(g)','D(g)','K(g)'])
axes[0].set_title('Spearman Correlation Between Components', fontweight='bold')

# Evidence depth bar chart
depth = pcbs_df['n_components'].value_counts().sort_index()
axes[1].bar(depth.index, depth.values, color='#1565C0', edgecolor='white')
axes[1].set_xlabel('Number of Evidence Components')
axes[1].set_ylabel('Gene Count')
axes[1].set_title('Evidence Depth per Gene', fontweight='bold')
axes[1].set_xticks(range(0, 6))
for idx, val in zip(depth.index, depth.values):
    axes[1].text(idx, val + max(depth.values)*0.01, f'{val:,}',
                ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('results/pcbs_component_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/pcbs_component_analysis.png')

print('\nComponent independence (Spearman rho):')
pairs = []
for i, c1 in enumerate(comp_cols):
    for j, c2 in enumerate(comp_cols):
        if j > i:
            r = corr_matrix.loc[c1, c2]
            pairs.append((c1, c2, r))
pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for c1, c2, r in pairs:
    interp = 'strong' if abs(r) > 0.5 else 'moderate' if abs(r) > 0.3 else 'weak'
    print(f'  {c1} vs {c2}: rho = {r:+.3f} ({interp})')

## 8 \u00b7 Known Gene Validation

Check whether established PCOS\u2013endometrial cancer bridge genes rank highly in the PCBS.

In [ ]:
known_bridge_genes = [
    'AKT1', 'PIK3CA', 'PTEN', 'ESR1', 'AR', 'INSR', 'IGF1R',
    'CYP19A1', 'CYP11A1', 'STAR', 'HSD3B2',
    'IL6', 'TNF', 'VEGFA', 'TP53',
    'ERBB2', 'ERBB3', 'FOXO1', 'CCND1', 'MYC',
]

known_results = []
for gene in known_bridge_genes:
    row = pcbs_df[pcbs_df['Gene'] == gene]
    if len(row) > 0:
        r = row.iloc[0]
        rank = int((pcbs_df['PCBS'] > r['PCBS']).sum() + 1)
        pct = 100 * (1 - rank / len(pcbs_df))
        known_results.append({
            'Gene': gene, 'PCBS': r['PCBS'],
            'P_g': r['P_g'], 'T_g': r['T_g'], 'C_g': r['C_g'],
            'D_g': r['D_g'], 'K_g': r['K_g'],
            'Rank': rank, 'Percentile': pct,
            'n_components': int(r['n_components']), 'Found': True
        })
    else:
        known_results.append({'Gene': gene, 'Found': False})

known_df = pd.DataFrame(known_results)
found = known_df[known_df['Found'] == True].copy()
missing = known_df[known_df['Found'] == False]

print(f'Known Bridge Gene Validation ({len(known_bridge_genes)} genes)')
print('=' * 85)
print(f'Found in scored genes: {len(found)}/{len(known_bridge_genes)}')
if len(missing) > 0:
    print(f'Not found: {  ", ".join(missing["Gene"].tolist())}')

if len(found) > 0:
    fs = found.sort_values('PCBS', ascending=False)
    print(f'\n{"Gene":<10} {"PCBS":>7} {"Rank":>6} {"Pctl":>7}  '
          f'{"P(g)":>6} {"T(g)":>6} {"C(g)":>6} {"D(g)":>6} {"K(g)":>6} {"#":>2}')
    print('-' * 85)
    for _, r in fs.iterrows():
        print(f'{r["Gene"]:<10} {r["PCBS"]:7.4f} {int(r["Rank"]):6d} '
              f'{r["Percentile"]:6.1f}%  {r["P_g"]:6.4f} {r["T_g"]:6.4f} '
              f'{r["C_g"]:6.4f} {r["D_g"]:6.4f} {r["K_g"]:6.4f} {int(r["n_components"]):2d}')

    median_pct = fs['Percentile'].median()
    top5  = (fs['Percentile'] >= 95).sum()
    top10 = (fs['Percentile'] >= 90).sum()
    print(f'\nMedian percentile of known genes: {median_pct:.1f}%')
    print(f'Known genes in top  5%: {top5}/{len(found)}')
    print(f'Known genes in top 10%: {top10}/{len(found)}')

## 9 \u00b7 Component Profile \u2014 Radar Chart

Radar plot showing which evidence components drive the top bridge genes.

In [ ]:
from math import pi

top8 = pcbs_df.nlargest(8, 'PCBS')
categories = ['P(g)', 'T(g)', 'C(g)', 'D(g)', 'K(g)']
N = len(categories)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
cmap = plt.cm.Set2

for i, (_, row) in enumerate(top8.iterrows()):
    values = [row['P_g'], row['T_g'], row['C_g'], row['D_g'], row['K_g']]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=1.5, label=row['Gene'],
            color=cmap(i / 8), markersize=4)
    ax.fill(angles, values, alpha=0.05, color=cmap(i / 8))

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylim(0, 1.05)
ax.set_title('Evidence Profile \u2014 Top 8 Bridge Genes',
             fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=9)

plt.tight_layout()
plt.savefig('results/pcbs_radar_top8.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/pcbs_radar_top8.png')

## 10 \u00b7 Weight Sensitivity Check

Verify that the top gene rankings are stable under alternative weighting schemes.

In [ ]:
# Alternative weighting schemes
alt_weights = {
    'Equal (0.20 each)':       [0.20, 0.20, 0.20, 0.20, 0.20],
    'Molecular-heavy':         [0.25, 0.25, 0.25, 0.15, 0.10],
    'Causal-heavy':            [0.15, 0.15, 0.15, 0.25, 0.30],
    'Transition-emphasis':     [0.20, 0.30, 0.20, 0.15, 0.15],
}

cols_order = ['P_g', 'T_g', 'C_g', 'D_g', 'K_g']
rank_results = {}

for scheme_name, wts in alt_weights.items():
    score = sum(pcbs_df[c] * w for c, w in zip(cols_order, wts))
    rank_results[scheme_name] = score.rank(ascending=False, method='min')

rank_df = pd.DataFrame(rank_results)

# Spearman correlation of RANKINGS across schemes
rank_corr = rank_df.corr(method='spearman')
print('Rank stability across weighting schemes (Spearman rho):')
print(rank_corr.to_string(float_format='%.4f'))

# Show top-10 overlap
print(f'\nTop 10 overlap across schemes:')
base_top10 = set(pcbs_df.nlargest(10, 'PCBS')['Gene'])
for scheme_name, wts in alt_weights.items():
    score = sum(pcbs_df[c] * w for c, w in zip(cols_order, wts))
    alt_top10 = set(pcbs_df.nlargest(10, score.name if hasattr(score,'name') else 0)['Gene'])
    # recompute properly
    pcbs_df['_tmp'] = score
    alt_top10 = set(pcbs_df.nlargest(10, '_tmp')['Gene'])
    overlap = len(base_top10 & alt_top10)
    print(f'  {scheme_name}: {overlap}/10 genes shared with equal-weight top 10')

pcbs_df.drop(columns='_tmp', inplace=True, errors='ignore')

## 11 \u00b7 Save Results

In [ ]:
save_cols = ['Gene', 'P_g', 'T_g', 'C_g', 'D_g', 'K_g', 'PCBS', 'n_components']

# Full scored gene list
pcbs_df[save_cols].to_csv('results/pcbs_scores.csv', index=False)
print(f'Saved: results/pcbs_scores.csv  ({len(pcbs_df):,} genes)')

# Bridge genes only (PCBS > 0)
bridge = (pcbs_df[pcbs_df['PCBS'] > 0]
          .sort_values('PCBS', ascending=False)
          .reset_index(drop=True))
bridge[save_cols].to_csv('results/pcbs_bridge_genes.csv', index=False)
print(f'Saved: results/pcbs_bridge_genes.csv  ({len(bridge):,} genes)')

# Top 5%
pct95 = pcbs_df['PCBS'].quantile(0.95)
top5 = (pcbs_df[pcbs_df['PCBS'] >= pct95]
        .sort_values('PCBS', ascending=False))
top5[save_cols].to_csv('results/pcbs_top5pct_genes.csv', index=False)
print(f'Saved: results/pcbs_top5pct_genes.csv  ({len(top5):,} genes, PCBS >= {pct95:.4f})')

# Known gene validation
if len(found) > 0:
    found.sort_values('PCBS', ascending=False).to_csv(
        'results/pcbs_known_gene_validation.csv', index=False)
    print(f'Saved: results/pcbs_known_gene_validation.csv  ({len(found)} genes)')

print(f'\n{"="*50}')
print(f'PCBS SCORING COMPLETE')
print(f'{"="*50}')
print(f'Total genes scored:        {len(pcbs_df):,}')
print(f'Genes with PCBS > 0:       {(pcbs_df["PCBS"] > 0).sum():,}')
print(f'Top 5% threshold:          {pct95:.4f}')
print(f'Max PCBS:                  {pcbs_df["PCBS"].max():.4f}'
      f'  ({pcbs_df.loc[pcbs_df["PCBS"].idxmax(), "Gene"]})')
print(f'Known genes found/checked: {len(found)}/{len(known_bridge_genes)}')
if len(found) > 0:
    print(f'Known genes median pctl:   {found["Percentile"].median():.1f}%')

## Summary

NB6 integrated five independent evidence sources into a unified PCBS score:

| Component | Source | Measures |
|-----------|--------|----------|
| P(g) | NB1 \u2014 GSE48301 | PCOS endometrial DEG evidence |
| T(g) | NB3 \u2014 GSE106191 | Hyperplasia\u2192cancer transition evidence |
| C(g) | NB2 \u2014 TCGA-UCEC | Endometrial cancer DEG evidence |
| D(g) | NB4 \u2014 GSE39099 | Cross-stage directional consistency |
| K(g) | NB5 \u2014 MR/GWAS | Independent causal knowledge |

**Next:** NB7 uses the scored genes + STRING PPI to construct the evidence-weighted graph.

In [ ]:
import zipfile

zip_path = 'results/NB6_PCBS_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in ['results/pcbs_scores.csv',
              'results/pcbs_bridge_genes.csv',
              'results/pcbs_top5pct_genes.csv',
              'results/pcbs_known_gene_validation.csv',
              'results/pcbs_distributions.png',
              'results/pcbs_top20_genes.png',
              'results/pcbs_component_analysis.png',
              'results/pcbs_radar_top8.png']:
        if os.path.exists(f):
            zf.write(f)

print(f'Results ZIP: {zip_path}')

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not running in Colab \u2014 download manually.')